# 05 — vLLM comparison (Phases 5 and 6)

Phase 5 makes measurement trustworthy; Phase 6 uses it against vLLM.

**Read this before running anything.** vLLM pins its own torch build and
installing it will replace the one the rest of LatentServe runs against.
So this notebook is in **two halves that must run in separate sessions**:

1. Part A — LatentServe rows, on the current environment.
2. *Restart the session*, install vLLM, run Part B.
3. Part C — compare, in either session.

Trying to run both engines in one process also fails for a second
reason: vLLM claims a large persistent share of VRAM at construction, so
LatentServe's cache would be sized against the leftovers.

**Budget:** ~10 min for Part A, ~15 min for Part B including the install.

In [ ]:
import os

# Set before torch is imported anywhere, and before any `!python` child
# process inherits the environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Two ways in. If you attached the repo as a Kaggle Dataset, point
# UPLOAD at it; otherwise it clones. A private repo needs a PAT in
# Add-ons -> Secrets as GITHUB_TOKEN.
REPO  = "https://github.com/Jash-Vora/LatentServe.git"
DEST  = Path("/kaggle/working/LatentServe")
UPLOAD = Path("/kaggle/input/latentserve")   # optional dataset fallback

if not DEST.exists():
    if UPLOAD.exists():
        !cp -r {UPLOAD} {DEST}
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass  # public repo, or you're pulling a fork
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        subprocess.run(["git", "clone", "--depth", "1", url, str(DEST)], check=True)

os.chdir(DEST)
sys.path.insert(0, str(DEST))
os.environ["PYTHONPATH"] = f"{DEST}:" + os.environ.get("PYTHONPATH", "")
print("cwd:", Path.cwd())
!git log --oneline -1

In [ ]:
# Do NOT install torch here. Kaggle ships a build matched to its driver;
# replacing it is the fastest way to lose an afternoon.
!pip install -q pydantic pyyaml 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
if (major, minor) < (4, 40):
    print("\n[!] pyproject requires transformers>=4.40. Run:  !pip install -q -U transformers")
else:
    print("transformers OK (LatentServe's own layer loop is version-independent "
          "by design, but model/qwen.py's HF cache walk needs >=4.40)")

In [ ]:
# Pull the checkpoint once, up front, so the download (~3 GB) never lands
# inside a timed benchmark. Later sessions re-download unless you save the
# HF cache as a Kaggle Dataset — worth doing once you're iterating daily.
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={cfg.num_hidden_layers} q_heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} "
      f"group={cfg.num_attention_heads // cfg.num_key_value_heads} "
      f"head_dim={getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads}")
_ = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
_ = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
del _
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("cached")

## 1. Phase 5 — the harness's own tests

No GPU, no model, no vLLM. These cover the two metric bugs that cost
Phase 4 its predictions, plus the fairness guard Phase 6 depends on.

In [ ]:
!python -m pytest tests/test_phase5_harness.py -v 2>&1 | tail -25

## 2. Establish the noise floor

Every ratio in the final report is a claim that a difference exceeds
run-to-run variation. This measures that variation directly, so "3.3%
faster" can be defended and "0.4% faster" can be called noise.

In [ ]:
import json, pathlib
from benchmarks.harness import repeatability_report, check_environment

# Re-uses the Phase 4 scheduler rows: same config run more than once.
path = pathlib.Path("results/raw/phase4_serving.jsonl")
if path.exists():
    rows = [json.loads(line) for line in path.open()]
    fifo = [r for r in rows if r["system"] == "latentserve_sched_fifo"]
    for metric in ("throughput_tokens_sec", "ttft_p50_ms"):
        print(repeatability_report(fifo, metric))
    drift = check_environment(rows)
    print("\nenvironment drift:", drift or "none — all rows from one commit/env")
else:
    print("Run notebook 04 first; this reads its results.")

## 3. Part A — LatentServe rows

Prompts are built once from the tokenizer and reused as raw token ids,
so no tokenizer difference can enter the comparison.

In [ ]:
!python -m benchmarks.runners.phase6_vllm \
    --config configs/phase6_vllm.yaml --system latentserve \
    --workload mixed --num-requests 32 \
    --max-prompt 8192 --max-output 256 --batch-sizes 4 8

## 4. Part B — vLLM rows (restart the session first)

Run the setup cells above again after the restart, then this one, then
the comparison. If the install picks a vLLM that has dropped Turing
(sm75) support, pin an older release rather than changing the model or
dtype — those are controlled variables and changing them invalidates the
comparison.

In [ ]:
# Install, then check the GPU is actually supported before spending time.
!pip install -q vllm 2>&1 | tail -3

import torch
from comparisons.vllm.runner import vllm_available
ok, version = vllm_available()
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
print(f"vllm installed: {ok} version={version}")
print(f"device capability: {cap}  (T4 = (7, 5), Turing)")
print(f"torch now: {torch.__version__}  <- vLLM may have replaced this")

In [ ]:
!VLLM_USE_V1=0 python -m benchmarks.runners.phase6_vllm \
    --config configs/phase6_vllm.yaml --system vllm \
    --workload mixed --num-requests 32 \
    --max-prompt 8192 --max-output 256 --batch-sizes 4 8

## 5. Part C — compare

`--compare` refuses any pair whose controlled variables differ, printing
`SKIPPED` with the mismatch rather than a plausible wrong number.

In [ ]:
!python -m benchmarks.runners.phase6_vllm --compare

In [ ]:
import pandas as pd

df = pd.read_json("results/raw/phase6_vllm.jsonl", lines=True)
df = pd.concat([df.drop(columns=["extra"]), pd.json_normalize(df["extra"])], axis=1)
cols = [c for c in ["system", "batch_size", "throughput_tokens_sec", "ttft_p50",
                    "itl_p50", "itl_stall_rate", "peak_vram_mb", "vllm_version",
                    "vllm_chunked_prefill", "vllm_enable_prefix_caching"]
        if c in df.columns]
print(df[cols].round(2).to_string(index=False))
print("\np50 inter-token latency is comparable across systems.")
print("p99 is NOT: vLLM reports a per-request mean, LatentServe reports per-token")
print("gaps, and Phase 4 showed those differ by ~30x under prefill blocking.")

## 6. Before you close the session

Gate 6 checklist is in `docs/phase6.md`. The important one is the last
item: **attribute every gap to a mechanism** rather than reporting it.
The three candidates, all already measured:

* the paged gather (Phase 3: 2x resident KV per step; Phase 4: TPOT
  33 -> 65 ms from batch 1 to 8) — this is what the Phase 11 kernel
  removes, so the gap at large batch estimates that kernel's value;
* CUDA graphs — Phase 2's P1 found decode at 24-32% of peak bandwidth,
  i.e. overhead-bound, and LatentServe runs eager Python per layer;
* chunked prefill — Phase 4 measured LatentServe's prefill blocking as
  ~2.3 s inter-token stalls at ~1% stall rate.

In [ ]:
!cd /kaggle/working/LatentServe && zip -qr /kaggle/working/results_phase6.zip results/raw
!ls -la /kaggle/working/*.zip